# TabDiff dataset overview

Scan `data/*/info.json` for bundled tabular datasets: **sizes**, **train/val/test splits** (from `info.json` counts and on-disk `.npy` files), and **feature layout** (numerical vs categorical columns and target).

Run this notebook from the **repository root** or from **`notebooks/`** (paths are resolved automatically).

In [10]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

# Repo root: cwd, or parent if cwd is notebooks/
_here = Path.cwd().resolve()
REPO_ROOT = _here if (_here / "data").is_dir() else _here.parent
DATA_DIR = REPO_ROOT / "data"
assert DATA_DIR.is_dir(), f"Expected data/ under {REPO_ROOT}"
print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = /cephfs/home/okashurin/TabDiff


In [11]:
def list_dataset_names(data_dir: Path) -> list[str]:
    """Subdirectories of data/ that contain info.json (excludes data/Info/)."""
    out = []
    for p in sorted(data_dir.iterdir()):
        if p.is_dir() and (p / "info.json").is_file():
            out.append(p.name)
    return out


def npy_splits_present(ds_path: Path) -> dict[str, bool]:
    """Which splits have TabDiff-style y_*.npy (and optional X_num/X_cat)."""
    splits = {}
    for split in ("train", "val", "test"):
        yp = ds_path / f"y_{split}.npy"
        splits[split] = yp.is_file()
    return splits


def npy_row_counts(ds_path: Path) -> dict[str, int | None]:
    """Row counts from y_*.npy when present."""
    counts: dict[str, int | None] = {}
    for split in ("train", "val", "test"):
        yp = ds_path / f"y_{split}.npy"
        if yp.is_file():
            y = np.load(yp, allow_pickle=True)
            counts[split] = int(y.shape[0])
        else:
            counts[split] = None
    return counts


def load_info(data_dir: Path, name: str) -> dict:
    with open(data_dir / name / "info.json") as f:
        return json.load(f)


def build_summary_table(data_dir: Path) -> pd.DataFrame:
    rows = []
    for name in list_dataset_names(data_dir):
        info = load_info(data_dir, name)
        ds_path = data_dir / name
        splits = npy_splits_present(ds_path)
        n_rows = npy_row_counts(ds_path)
        n_num = len(info.get("num_col_idx", []) or [])
        n_cat = len(info.get("cat_col_idx", []) or [])
        n_tgt = len(info.get("target_col_idx", []) or [])
        rows.append(
            {
                "dataname": name,
                "task_type": info.get("task_type", ""),
                "train_n (info)": info.get("train_num"),
                "val_n (info)": info.get("val_num"),
                "test_n (info)": info.get("test_num"),
                "train_n (npy)": n_rows["train"],
                "val_n (npy)": n_rows["val"],
                "test_n (npy)": n_rows["test"],
                "npy train": splits["train"],
                "npy val": splits["val"],
                "npy test": splits["test"],
                "#num_cols": n_num,
                "#cat_cols": n_cat,
                "#target_cols": n_tgt,
                "#features (excl. target)": n_num + n_cat,
                "dcr_split": "_dcr" in name,
            }
        )
    return pd.DataFrame(rows)


summary = build_summary_table(DATA_DIR)
summary

,dataname,task_type,train_n (info),val_n (info),test_n (info),train_n (npy),val_n (npy),test_n (npy),npy train,npy val,npy test,#num_cols,#cat_cols,#target_cols,#features (excl. target),dcr_split
0,adult,binclass,32561,0,16281,32561,NaN,16281,True,False,True,6,8,1,14,False
1,adult_dcr,binclass,24421,0,24421,24421,NaN,24421,True,False,True,6,8,1,14,True
2,beijing,regression,37581,0,4176,37581,NaN,4176,True,False,True,6,5,1,11,False
3,beijing_dcr,regression,20878,0,20879,20878,NaN,20879,True,False,True,6,5,1,11,True
4,default,binclass,27000,0,3000,27000,NaN,3000,True,False,True,14,9,1,23,False
5,default_dcr,binclass,15000,0,15000,15000,NaN,15000,True,False,True,14,9,1,23,True
6,diabetes,binclass,61059,20353,20354,61059,20353.0,20354,True,True,True,9,27,1,36,False
7,diabetes_dcr,binclass,50883,0,50883,50883,0.0,50883,True,True,True,9,27,1,36,True
8,magic,binclass,17117,0,1902,17117,NaN,1902,True,False,True,10,0,1,10,False
9,news,regression,35679,0,3965,35679,NaN,3965,True,False,True,45,2,1,47,False


### Optional: styled view
Uncomment if you use Jupyter with pandas Styler support.

In [12]:
# summary.style.background_gradient(subset=["train_n (npy)", "test_n (npy)"], cmap="Blues")

## Feature list for one dataset
Set `DATANAME` to any row from the table above (e.g. `"adult"`, `"news_dcr"`).

In [14]:
from IPython.display import display

DATANAME = "adult"


def categorical_train_cardinality(dataset_dir: Path, info: dict) -> dict[int, int]:
    """Unique values per categorical column on train `.npy` (same idea as `src.get_categories`)."""
    out: dict[int, int] = {}
    cat_col_idx = list(info.get("cat_col_idx") or [])
    xcat = dataset_dir / "X_cat_train.npy"
    if xcat.is_file():
        Xc = np.load(xcat, allow_pickle=True)
        if Xc.ndim == 2:
            for j, col_i in enumerate(cat_col_idx):
                if j < Xc.shape[1]:
                    out[int(col_i)] = int(len(np.unique(Xc[:, j])))
    task_type = info.get("task_type") or ""
    if task_type != "regression":
        yp = dataset_dir / "y_train.npy"
        if yp.is_file():
            y = np.asarray(np.load(yp, allow_pickle=True))
            if y.ndim == 1:
                y = y.reshape(-1, 1)
            for ti, col_i in enumerate(info.get("target_col_idx") or []):
                if ti < y.shape[1]:
                    out[int(col_i)] = int(len(np.unique(y[:, ti])))
    return out


def describe_dataset(data_dir: Path, name: str) -> None:
    info = load_info(data_dir, name)
    cols = info.get("column_names") or []
    num_idx = set(info.get("num_col_idx", []) or [])
    cat_idx = set(info.get("cat_col_idx", []) or [])
    tgt_idx = set(info.get("target_col_idx") or [])
    int_idx = set(info.get("int_col_idx", []) or [])
    task_type = info.get("task_type") or ""
    cards = categorical_train_cardinality(data_dir / name, info)
    col_info = info.get("column_info") or {}

    rows = []
    for i, cname in enumerate(cols):
        role = []
        if i in tgt_idx:
            role.append("target")
        if i in num_idx:
            role.append("numerical")
        if i in cat_idx:
            role.append("categorical")
        if i in int_idx:
            role.append("integer-like")
        is_cat_feat = i in cat_idx or (i in tgt_idx and task_type != "regression")
        if not is_cat_feat:
            card = pd.NA
        elif i in cards:
            card = cards[i]
        else:
            ent = col_info.get(str(i))
            if isinstance(ent, dict) and "categorizes" in ent:
                card = len(ent["categorizes"])
            else:
                card = pd.NA
        rows.append(
            {
                "idx": i,
                "column": cname,
                "roles": ", ".join(role) or "—",
                "cardinality": card,
            }
        )
    feat_df = pd.DataFrame(rows)
    if "cardinality" in feat_df.columns:
        feat_df["cardinality"] = feat_df["cardinality"].astype("Int64")

    print(f"=== {name} ===")
    print("task_type:", info.get("task_type"))
    print("train_num / val_num / test_num (info.json):", info.get("train_num"), "/", info.get("val_num"), "/", info.get("test_num"))
    print("npy splits:", npy_splits_present(data_dir / name))
    print("npy row counts:", npy_row_counts(data_dir / name))
    print()
    display(feat_df)

    # Synthetic eval paths (used by TabDiff metrics) — may be missing until you generate them
    syn = REPO_ROOT / "synthetic" / name
    if syn.is_dir():
        files = sorted(p.name for p in syn.iterdir() if p.is_file())
        print("synthetic/ files (sample):", files[:20], "..." if len(files) > 20 else "")
    else:
        print("No synthetic/", name, "(metrics expect synthetic/<dataname>/real.csv, test.csv, val.csv when present)")


describe_dataset(DATA_DIR, DATANAME)

=== adult ===
task_type: binclass
train_num / val_num / test_num (info.json): 32561 / 0 / 16281
npy splits: {'train': True, 'val': False, 'test': True}
npy row counts: {'train': 32561, 'val': None, 'test': 16281}



,idx,column,roles,cardinality
0,0,age,"numerical, integer-like",<NA>
1,1,workclass,categorical,9
2,2,fnlwgt,"numerical, integer-like",<NA>
3,3,education,categorical,16
4,4,education.num,"numerical, integer-like",<NA>
5,5,marital.status,categorical,7
6,6,occupation,categorical,15
7,7,relationship,categorical,6
8,8,race,categorical,5
9,9,sex,categorical,2


synthetic/ files (sample): ['real.csv', 'test.csv'] 


## CSV files on disk (raw / auxiliary)
TabDiff training reads processed `.npy` splits; CSVs are often the raw source or extras.

In [15]:
def list_csvs(ds_path: Path, max_list: int = 30) -> pd.DataFrame:
    csvs = sorted(ds_path.rglob("*.csv"))
    rows = []
    for p in csvs[:max_list]:
        rel = p.relative_to(ds_path)
        try:
            n = sum(1 for _ in open(p, "rb")) - 1  # header
        except OSError:
            n = None
        rows.append({"path": str(rel), "approx_rows (excl header)": n})
    df = pd.DataFrame(rows)
    if len(csvs) > max_list:
        print(f"Showing first {max_list} of {len(csvs)} CSV files")
    return df


list_csvs(DATA_DIR / DATANAME)

,path,approx_rows (excl header)
0,test.csv,16281
1,train.csv,32561
